In [1]:
!nvidia-smi

Mon Mar  2 12:10:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.8 MB/s eta 0:00:00a 0:00:01


In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import kneighbors_graph, NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import classification_report
from scipy.sparse import lil_matrix
import time

CSV_FILE_PATH = '/kaggle/input/datasets/ayyuce/brca-subtypes/brca_data_w_subtypes 2.csv'
TARGET_COLS = ['histological.type', 'vital.status', 'PR.Status', 'ER.Status', 'HER2.Final.Status']
KNN_K = 10
HIDDEN_CHANNELS = 256
LEARNING_RATE = 0.001
EPOCHS = 100

def load_and_preprocess_data():
    try:
        df = pd.read_csv(CSV_FILE_PATH)
        feature_cols = [col for col in df.columns if col.startswith(('rs_', 'cn_', 'mu_', 'pp_'))]
        
        if df[feature_cols].isnull().sum().sum() > 0:
            numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns
            df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

        for col in ['PR.Status', 'ER.Status', 'HER2.Final.Status']:
            if col in df.columns:
                df[col] = df[col].astype(str).replace({
                    'nan': 'Unknown', 'Indeterminate': 'Unknown', 'Not Performed': 'Unknown'
                })

        if 'vital.status' in df.columns:
            df['vital.status'] = df['vital.status'].astype(str).str.lower()
            df['vital.status'] = df['vital.status'].replace({'alive': '0', 'dead': '1'})
            df['vital.status'] = pd.to_numeric(df['vital.status'], errors='coerce').fillna(0).astype(int)

        for col in TARGET_COLS:
            if col not in df.columns:
                continue
            label_counts = df[col].value_counts()
            max_samples = label_counts.max()
            df_resampled = []
            for class_val in df[col].unique():
                df_class = df[df[col] == class_val]
                if len(df_class) < max_samples:
                    df_resampled.append(resample(df_class, replace=True, n_samples=max_samples, random_state=42))
                else:
                    df_resampled.append(df_class)
            df = pd.concat(df_resampled, ignore_index=True)

        X_raw = df[feature_cols].values
        y_dict, label_map = {}, {}
        for col in TARGET_COLS:
            if col in df.columns:
                le = LabelEncoder()
                y_dict[col] = le.fit_transform(df[col].astype(str))
                label_map[col] = le.classes_

        return X_raw, y_dict, label_map
    except Exception:
        return None, None, None

class GCNPart1(torch.nn.Module):
    def __init__(self, num_features, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        return F.dropout(x, p=0.3, training=self.training)

class GCNPart2(torch.nn.Module):
    def __init__(self, hidden_channels, class_counts):
        super().__init__()
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        
        self.heads = torch.nn.ModuleDict()
        self.active_cols = list(class_counts.keys())
        for col, num_classes in class_counts.items():
            self.heads[col.replace('.', '_')] = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        
        return [self.heads[col.replace('.', '_')](x) for col in self.active_cols]

X_raw, y_dict, label_map = load_and_preprocess_data()

if X_raw is not None:
    indices = np.arange(len(X_raw))
    train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

    scaler = StandardScaler()
    X_scaled = np.zeros_like(X_raw, dtype=np.float32)
    X_scaled[train_idx] = scaler.fit_transform(X_raw[train_idx])
    X_scaled[val_idx] = scaler.transform(X_raw[val_idx])
    X_scaled[test_idx] = scaler.transform(X_raw[test_idx])

    train_graph = kneighbors_graph(X_scaled[train_idx], n_neighbors=KNN_K, mode='connectivity', include_self=False)
    nbrs = NearestNeighbors(n_neighbors=KNN_K, algorithm='auto').fit(X_scaled[train_idx])
    _, val_neighbors = nbrs.kneighbors(X_scaled[val_idx])
    _, test_neighbors = nbrs.kneighbors(X_scaled[test_idx])

    n_train, n_val, n_test = len(train_idx), len(val_idx), len(test_idx)
    n_total = n_train + n_val + n_test

    full_adj = lil_matrix((n_total, n_total))
    full_adj[:n_train, :n_train] = train_graph
    for i, neighbors in enumerate(val_neighbors):
        for neighbor in neighbors:
            full_adj[n_train + i, neighbor] = 1
    for i, neighbors in enumerate(test_neighbors):
        for neighbor in neighbors:
            full_adj[n_train + n_val + i, neighbor] = 1

    full_adj = full_adj.tocsr()
    full_adj = full_adj + full_adj.T
    edge_index_np = np.array(full_adj.nonzero())

    gpu0 = torch.device('cuda:0' if torch.cuda.device_count() > 0 else 'cpu')
    gpu1 = torch.device('cuda:1' if torch.cuda.device_count() > 1 else gpu0)

    x_tensor = torch.from_numpy(X_scaled).float()
    edge_index = torch.from_numpy(edge_index_np).long()
    
    active_cols = [col for col in TARGET_COLS if col in y_dict]
    labels = {col: torch.from_numpy(y_dict[col]).long() for col in active_cols}

    def create_mask(idx_list):
        mask = torch.zeros(n_total, dtype=torch.bool)
        mask[idx_list] = True
        return mask

    train_mask = create_mask(train_idx).to(gpu1)
    val_mask = create_mask(val_idx).to(gpu1)
    test_mask = create_mask(test_idx).to(gpu1)

    x_0 = x_tensor.to(gpu0)
    edge_index_0 = edge_index.to(gpu0)
    edge_index_1 = edge_index.to(gpu1)
    
    labels_1 = {col: val.to(gpu1) for col, val in labels.items()}

    class_counts = {col: len(label_map[col]) for col in active_cols}
    
    model_part1 = GCNPart1(X_scaled.shape[1], HIDDEN_CHANNELS).to(gpu0)
    model_part2 = GCNPart2(HIDDEN_CHANNELS, class_counts).to(gpu1)

    optimizer = torch.optim.Adam(
        list(model_part1.parameters()) + list(model_part2.parameters()), 
        lr=LEARNING_RATE
    )

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        
        model_part1.train()
        model_part2.train()
        
        optimizer.zero_grad()

        out_gpu0 = model_part1(x_0, edge_index_0)
        
        out_to_gpu1 = out_gpu0.to(gpu1)
        
        final_outputs = model_part2(out_to_gpu1, edge_index_1)
        
        loss = sum(F.cross_entropy(final_outputs[i][train_mask], labels_1[col][train_mask]) 
                   for i, col in enumerate(active_cols))
        
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | Train Loss: {loss.item():.4f} | Time: {time.time() - start_time:.2f}s")

    model_part1.eval()
    model_part2.eval()
    
    with torch.no_grad():
        out_gpu0_test = model_part1(x_0, edge_index_0)
        final_outputs_test = model_part2(out_gpu0_test.to(gpu1), edge_index_1)
        
        for i, col in enumerate(active_cols):
            preds = final_outputs_test[i][test_mask].argmax(dim=1).cpu().numpy()
            true_vals = labels_1[col][test_mask].cpu().numpy()
            unique_classes = np.unique(np.concatenate([true_vals, preds]))
            active_labels = [label_map[col][c] for c in unique_classes]
            
            print(f"\nResults for {col}:")
            print(classification_report(true_vals, preds, labels=unique_classes, target_names=active_labels, zero_division=0))

Epoch 010 | Train Loss: 4.9378 | Time: 0.05s
Epoch 020 | Train Loss: 4.5687 | Time: 0.04s
Epoch 030 | Train Loss: 4.3437 | Time: 0.04s
Epoch 040 | Train Loss: 4.1592 | Time: 0.04s
Epoch 050 | Train Loss: 3.9642 | Time: 0.04s
Epoch 060 | Train Loss: 3.7661 | Time: 0.04s
Epoch 070 | Train Loss: 3.5298 | Time: 0.04s
Epoch 080 | Train Loss: 3.2790 | Time: 0.04s
Epoch 090 | Train Loss: 3.0119 | Time: 0.04s
Epoch 100 | Train Loss: 2.7736 | Time: 0.04s

Results for histological.type:
                                precision    recall  f1-score   support

 infiltrating ductal carcinoma       0.88      0.98      0.93      1339
infiltrating lobular carcinoma       0.64      0.19      0.30       231

                      accuracy                           0.87      1570
                     macro avg       0.76      0.59      0.61      1570
                  weighted avg       0.84      0.87      0.83      1570


Results for vital.status:
              precision    recall  f1-score   support

 